## resize mask
1. 对mask进行resize，观察在不同比例resize下，mask的显示效果
2. 在对mask进行resize之前，先见mask预处理成统一的格式（比如标签值统一为1或255）

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from collections import  Counter
import SimpleITK as sitk
import nibabel as nib
import time

In [2]:
def sitk_new_blank_image(size, spacing, direction, origin, default_value=0.):
    image = sitk.GetImageFromArray(np.ones(size, dtype=np.float).T * default_value)
    image.SetSpacing(spacing)
    image.SetDirection(direction)
    image.SetOrigin(origin)
    return image

def sitk_resample_to_image(image, reference_image, interpolator, default_value=0., transform=None,
                           output_pixel_type=None):
    if transform is None:
        transform = sitk.Transform()
        transform.SetIdentity()
    if output_pixel_type is None:
        output_pixel_type = image.GetPixelID()
    resample_filter = sitk.ResampleImageFilter()
    resample_filter.SetInterpolator(interpolator)
    resample_filter.SetTransform(transform)
    resample_filter.SetOutputPixelType(output_pixel_type)
    resample_filter.SetDefaultPixelValue(default_value)
    resample_filter.SetReferenceImage(reference_image)
    return resample_filter.Execute(image)

def generate_data_propotional(series_uid, scale, interpolator, output_pixel_type):
    reader = sitk.ImageSeriesReader()
    filenames = reader.GetGDCMSeriesFileNames(series_uid)
    reader.SetFileNames(filenames)
    im = reader.Execute()
    ori_spacing = im.GetSpacing()
    ori_size = im.GetSize()
    new_size = [0,0,0]
    for i in range(3):
        new_size[i] = int(ori_spacing[i]*ori_size[i]/(ori_spacing[0]))
    new_spacing = [ori_spacing[0]]*3

#     interpolator = sitk.sitkNearestNeighbor

    black_im = sitk_new_blank_image(new_size, new_spacing, im.GetDirection(), im.GetOrigin())
    new_im = sitk_resample_to_image(im, black_im, interpolator, default_value=0, output_pixel_type=output_pixel_type)

    return new_im

def generate_volume_data(series_uid, mask_out_path, modality):
    print('====> begin process {}'.format(series_uid))  
    beg = time.time()
    mask_name = os.path.basename(series_uid)
    
    im = None
    if modality == 'volume':
        im = generate_data_propotional(series_uid, None, sitk.sitkLinear, sitk.sitkInt16)
        mask_name = 'volume'
    elif modality == 'mask':
        im = generate_data_propotional(series_uid, None, sitk.sitkNearestNeighbor, sitk.sitkUInt8)
    if im is None:
        return
    
    # generate *.nii format
    sitk.WriteImage(im, os.path.join(mask_out_path, '{}.nii'.format(mask_name)))

    # generate *.nii.gz format
    sitk.WriteImage(im, os.path.join(mask_out_path, '{}.nii.gz'.format(mask_name)))
    
    # generate *.npy format
    np_img = sitk.GetArrayFromImage(im)
    with open(os.path.join(mask_out_path, '{}.npy'.format(mask_name)), 'wb') as f:
        np.save(f, np_img)
    end = time.time()
    print('Time elapsed:\t{:.3f}'.format(end-beg))
    print('====> end process {}\n'.format(mask_in_series))

In [3]:
# series_uid = '../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/bone'
# interpolator = sitk.sitkNearestNeighbor
# output_pixel_type = sitk.sitkUInt8
# new_im = generate_data_propotional(series_uid,None, interpolator, output_pixel_type)
# np_array = sitk.GetArrayFromImage(new_im)
# img = np_array.flatten()
# counter = Counter(img)

In [4]:
data_in_root = '../data/Liver'
data_out_root = '../data/processed_liver/scalex1'
# 生成文件，需要重新生成的时候，请解开注释
for sub_root_name in os.listdir(data_in_root):
    sub_root = os.path.join(data_in_root, sub_root_name)
    if not os.path.isdir(sub_root):
        continue
    if '3Dircadb' not in sub_root:
        continue
    mask_in_series = os.path.join(sub_root, 'MASKS_DICOM/MASKS_DICOM')
    if not os.path.isdir(mask_in_series):
        print('mask label not exist:\t{}'.format(mask_in_series))
        continue
    labels = os.listdir(mask_in_series)
    for label in labels:
        label_path = os.path.join(mask_in_series, label)
        if not os.path.isdir(label_path):
            continue
        mask_out_path = '../data/processed_liver/{}/scalex1/mask'.format(sub_root_name)
        os.makedirs(mask_out_path, exist_ok=True)
        generate_volume_data(label_path, mask_out_path, 'mask')
    
    volume_in_series = os.path.join(sub_root, 'PATIENT_DICOM/PATIENT_DICOM')
    if not os.path.isdir(volume_in_series):
        print('volume not exist:\t{}'.format(volume_in_series))
        continue
    volume_out_path = '../data/processed_liver/{}/scalex1/volume'.format(sub_root_name)
    os.makedirs(volume_out_path, exist_ok=True)
    generate_volume_data(volume_in_series, volume_out_path, 'volume')
    

====> begin process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/bone
Time elapsed:	15.887
====> end process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/portalvein1
Time elapsed:	18.881
====> end process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/livertumor
Time elapsed:	19.895
====> end process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/skin
Time elapsed:	18.781
====> end process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/venacava
Time elapsed:	17.390
====> end process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.10/MASKS_DICOM/MASKS_DICOM/liver
Time elapsed:	18.632
====> end process ../data/Liver/3Dircadb1.10/MASKS

Time elapsed:	15.287
====> end process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightsurretumor
Time elapsed:	18.579
====> end process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/artery
Time elapsed:	15.288
====> end process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/spleen
Time elapsed:	16.556
====> end process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/pancreas
Time elapsed:	18.630
====> end process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM/rightkidney
Time elapsed:	18.553
====> end process ../data/Liver/3Dircadb1.5/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.5/MASKS_DICOM/M

Time elapsed:	22.720
====> end process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor07
Time elapsed:	25.952
====> end process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/livertumor05
Time elapsed:	25.943
====> end process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/leftlung
Time elapsed:	23.821
====> end process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/spleen
Time elapsed:	24.194
====> end process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM/rightkidney
Time elapsed:	22.455
====> end process ../data/Liver/3Dircadb1.1/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.1/MASKS_DICO

Time elapsed:	16.404
====> end process ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/liver
Time elapsed:	25.149
====> end process ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM/portalvein
Time elapsed:	22.848
====> end process ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.4/PATIENT_DICOM/PATIENT_DICOM
Time elapsed:	31.365
====> end process ../data/Liver/3Dircadb1.4/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/bone
Time elapsed:	34.299
====> end process ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/venoussystem
Time elapsed:	22.135
====> end process ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.7/MASKS_DICOM/MASKS_DICOM/r

Time elapsed:	15.907
====> end process ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM/portalvein
Time elapsed:	14.837
====> end process ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.16/PATIENT_DICOM/PATIENT_DICOM
Time elapsed:	29.725
====> end process ../data/Liver/3Dircadb1.16/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/bone
Time elapsed:	12.950
====> end process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/livertumor
Time elapsed:	11.078
====> end process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM/skin
Time elapsed:	12.460
====> end process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASKS_DICOM

====> begin process ../data/Liver/3Dircadb1.13/MASKS_DICOM/MASK

In [6]:
print(counter)

NameError: name 'counter' is not defined